In [25]:
import pandas as pd


In [26]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [27]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [28]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [29]:
# vocab
vocab = {'<UNK>':0}

In [30]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [31]:
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [32]:
len(vocab)

324

In [33]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  
  return indexed_text

In [34]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [35]:
import torch
from torch.utils.data import Dataset, DataLoader

In [36]:
class QADasaset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab
  
  def __len__(self):
    return self.df.shape[0]
  
  def __getitem__(self, index):
    
    numercial_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numercial_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numercial_question), torch.tensor(numercial_answer)
    

In [37]:
dataset = QADasaset(df, vocab)

In [38]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [39]:
for question , answer in dataloader:
  print(question, answer[0])

tensor([[ 42, 101,   2,   3,  17]]) tensor([102])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[ 10,  96,   3, 104, 239]]) tensor([240])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([276])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([100])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([136])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([121])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[ 10,  75, 111]]) tensor([112])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([321]

In [40]:
import torch.nn as nn

In [41]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [42]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50,64,batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c,d= y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))
print("shape of e:", e.shape)


shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [43]:
learning_rate = 0.001
epochs=20

In [44]:
model = SimpleRNN(len(vocab))

In [45]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [47]:
# training loop
for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss:{total_loss:4f}")

Epoch: 1, Loss:519.065382
Epoch: 2, Loss:449.293245
Epoch: 3, Loss:371.439913
Epoch: 4, Loss:311.292722
Epoch: 5, Loss:260.495220
Epoch: 6, Loss:212.691332
Epoch: 7, Loss:169.389752
Epoch: 8, Loss:133.081013
Epoch: 9, Loss:102.279478
Epoch: 10, Loss:79.046268
Epoch: 11, Loss:61.412764
Epoch: 12, Loss:47.913983
Epoch: 13, Loss:38.084766
Epoch: 14, Loss:30.714085
Epoch: 15, Loss:25.357458
Epoch: 16, Loss:20.727746
Epoch: 17, Loss:17.488410
Epoch: 18, Loss:14.767671
Epoch: 19, Loss:12.741366
Epoch: 20, Loss:10.945518


In [48]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [49]:
predict(model, "What is the largest planet in our solar system?")

jupiter
